# Генерация текста

Свой LSTM, не библиотечный RNN. Готового модуля не было.
Два слоя, `lstm1` и `lstm2`. У каждого вентиля свой слой нейронов: веса от входа и от скрытого состояния.
Вентили: забывание, вход, память, выход. На каждом шаге сеть выбирает следующий символ.


In [2]:
import random
from pathlib import Path

import numpy as np
from numpy.random import randn

ROOT = Path.cwd().resolve()
if not (ROOT / "core").is_dir() and (ROOT.parent / "core").is_dir():
    ROOT = ROOT.parent


In [3]:
def sigmoid(value):
    return 1 / (1 + np.exp(-np.clip(value, -20, 20)))


class Layer:
    def __init__(self, n_in, n_out):
        self.W = randn(n_out, n_in) / np.sqrt(n_in)
        self.b = np.zeros((n_out, 1))


class LSTM:
    def __init__(self, input_size, hidden_size):
        joined = input_size + hidden_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.forget = Layer(joined, hidden_size)
        self.input = Layer(joined, hidden_size)
        self.cell = Layer(joined, hidden_size)
        self.output = Layer(joined, hidden_size)
        self.forget.b[:] = 1
        self.gates = (self.forget, self.input, self.cell, self.output)

    def step(self, x, h, c):
        joined = np.vstack([x, h])
        pre = [gate.W @ joined + gate.b for gate in self.gates]
        f = sigmoid(pre[0])
        i = sigmoid(pre[1])
        g = np.tanh(pre[2])
        o = sigmoid(pre[3])
        c_new = f * c + i * g
        h_new = o * np.tanh(c_new)
        return h_new, c_new, (joined, f, i, g, o, c_new, h_new, c)


class LSTM2:
    def __init__(self, input_size, output_size, hidden_size=32):
        self.hidden_size = hidden_size
        self.lstm1 = LSTM(input_size, hidden_size)
        self.lstm2 = LSTM(hidden_size, hidden_size)
        self.layers = (self.lstm1, self.lstm2)
        self.Why = randn(output_size, hidden_size) / np.sqrt(hidden_size)
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        c = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        self.cache = []
        ys = []
        for x in inputs:
            step_cache = []
            current = x
            for index, layer in enumerate(self.layers):
                h[index], c[index], cached = layer.step(current, h[index], c[index])
                step_cache.append(cached)
                current = h[index]
            self.cache.append(step_cache)
            ys.append(self.Why @ h[1] + self.by)
        self.last_h = h[1]
        return ys[-1]

    def backward(self, d_y, learn_rate=0.15):
        if isinstance(d_y, list):
            d_ys = d_y
        else:
            d_ys = [np.zeros_like(d_y) for _ in self.cache]
            d_ys[-1] = d_y
        d_Why = np.zeros_like(self.Why)
        d_by = np.zeros_like(self.by)
        buckets = []
        for layer in self.layers:
            buckets.append({
                gate: {"W": np.zeros_like(gate.W), "b": np.zeros_like(gate.b)}
                for gate in layer.gates
            })
        dh_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        dc_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        for t in reversed(range(len(self.cache))):
            dh = [item.copy() for item in dh_next]
            dc = [item.copy() for item in dc_next]
            joined, f, i, g, o, c_new, h_new, c_prev = self.cache[t][1]
            grad = d_ys[t]
            d_Why += grad @ h_new.T
            d_by += grad
            dh[1] = dh[1] + self.Why.T @ grad
            for li in (1, 0):
                layer = self.layers[li]
                joined, f, i, g, o, c_new, h_new, c_prev = self.cache[t][li]
                tanh_c = np.tanh(c_new)
                d_h = dh[li]
                d_c = d_h * o * (1 - tanh_c ** 2) + dc[li]
                d_pre = [
                    (d_c * c_prev) * f * (1 - f),
                    (d_c * g) * i * (1 - i),
                    (d_c * i) * (1 - g ** 2),
                    (d_h * tanh_c) * o * (1 - o),
                ]
                d_joined = np.zeros_like(joined)
                for gate, dz in zip(layer.gates, d_pre):
                    buckets[li][gate]["W"] += dz @ joined.T
                    buckets[li][gate]["b"] += dz
                    d_joined += gate.W.T @ dz
                dh_next[li] = d_joined[layer.input_size:]
                dc_next[li] = d_c * f
                if li == 1:
                    dh[0] = dh[0] + d_joined[:layer.input_size]
        np.clip(d_Why, -5, 5, out=d_Why)
        np.clip(d_by, -5, 5, out=d_by)
        for layer, bucket in zip(self.layers, buckets):
            for gate, grad in bucket.items():
                np.clip(grad["W"], -5, 5, out=grad["W"])
                np.clip(grad["b"], -5, 5, out=grad["b"])
                gate.W -= learn_rate * grad["W"]
                gate.b -= learn_rate * grad["b"]
        self.Why -= learn_rate * d_Why
        self.by -= learn_rate * d_by


def softmax(xs):
    z = xs - np.max(xs)
    e = np.exp(z)
    return e / np.sum(e)


In [4]:

corpus_path = ROOT / "data" / "kizaru_lyrics.txt"
if corpus_path.exists():
    text = corpus_path.read_text(encoding="utf-8")[:2500]
    print("corpus", corpus_path.name, len(text))
else:
    text = (
        "ночь город огонь и дым над рекой. "
        "я снова здесь и свет в окне чужой. "
        "биты стучат, пока район не спит. "
        "держи темп, не отпускай свой ритм. "
    ) * 8
    print("practice", len(text))

chars = sorted(set(text))
char_to_idx = {char: index for index, char in enumerate(chars)}
idx_to_char = {index: char for index, char in enumerate(chars)}
print("vocab", len(chars))


corpus kizaru_lyrics.txt 2500
vocab 99


In [5]:

def one_hot(index, size):
    vector = np.zeros((size, 1))
    vector[index] = 1
    return vector

SEQ = 36
np.random.seed(1)
random.seed(1)
net = LSTM2(len(chars), len(chars), hidden_size=24)

def step_language(start, learn_rate=0.2):
    chunk = text[start:start + SEQ + 1]
    inputs = [one_hot(char_to_idx[char], len(chars)) for char in chunk[:-1]]
    net.forward(inputs)
    grads = []
    loss = 0.0
    for t, char in enumerate(chunk[1:]):
        hidden = net.cache[t][1][6]
        logit = net.Why @ hidden + net.by
        probs = softmax(logit)
        target = char_to_idx[char]
        loss -= float(np.log(np.ravel(probs)[target] + 1e-9))
        grad = probs.copy()
        grad[target] -= 1
        grads.append(grad)
    net.backward(grads, learn_rate=learn_rate)
    return loss / SEQ

probe = 0
print("probe loss %.4f" % step_language(probe, learn_rate=0.0))
for epoch in range(12):
    loss = 0.0
    chunks = 0
    for start in range(0, 240, SEQ):
        loss += step_language(start, learn_rate=0.15)
        chunks += 1
    print("pass %d loss %.4f" % (epoch + 1, loss / chunks))

def sample(start_text, length, temperature=0.8):
    hidden = [np.zeros((net.hidden_size, 1)) for _ in net.layers]
    state = [np.zeros((net.hidden_size, 1)) for _ in net.layers]
    for char in start_text[:-1]:
        current = one_hot(char_to_idx.get(char, 0), len(chars))
        for index, layer in enumerate(net.layers):
            hidden[index], state[index], _ = layer.step(current, hidden[index], state[index])
            current = hidden[index]
    current_char = start_text[-1]
    output = start_text
    for _ in range(length):
        current = one_hot(char_to_idx.get(current_char, 0), len(chars))
        for index, layer in enumerate(net.layers):
            hidden[index], state[index], _ = layer.step(current, hidden[index], state[index])
            current = hidden[index]
        probs = softmax((net.Why @ hidden[1] + net.by) / temperature).ravel()
        pick = int(np.random.choice(len(chars), p=probs))
        current_char = idx_to_char[pick]
        output += current_char
    return output

random.seed(3)
print(sample("я", 100))


probe loss 4.6009
pass 1 loss 4.3248
pass 2 loss 3.9624
pass 3 loss 3.8239
pass 4 loss 3.7821
pass 5 loss 3.7487
pass 6 loss 3.7303
pass 7 loss 3.7153
pass 8 loss 3.7021
pass 9 loss 3.6899
pass 10 loss 3.6787
pass 11 loss 3.6602
pass 12 loss 3.6493
яфaйtrннынйи модч н.rхйеПweсnebсча лnnче  но arтnчаraaи ююерfнsаnneeьуa rrвnAa  aт  bнeeйну,nятnkвeнч
